# BIFlow — Étape 2 : Classification par Règles

**Owner**: Samar / Malek (Step 2)
**Branche**: `feature/semantic-detection-step2-rule-based`

Ce notebook démontre les 4 composants de l'Étape 2 :

1. **Schema Reader** — Lecture des schémas CSV et JSON
2. **Name Normalizer** — Normalisation des noms de colonnes
3. **Rule-Based Classifier** — Classification selon la taxonomie Step 1
4. **Ambiguity Handler** — Gestion des inconnus et des conflits

---

In [ ]:
# Setup — add project root to path if running from notebooks/ directory
import sys
from pathlib import Path

# Ensure the BIFlow project root is on sys.path
root = Path.cwd()
while root.name and not (root / 'contracts').exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f'Project root: {root}')

## 1. Schema Reader

Lecture d'un fichier CSV et extraction des métadonnées de colonnes.

In [ ]:
from intake.semantic_detection.step2.schema_reader import read_csv_schema, read_json_schema, read_schema_dir

# -- Example 1: read a CSV file (header + sample values)
example_dir = root / 'intake' / 'semantic_detection' / 'examples'

# Check available example files
print('Available example files:')
for f in sorted(example_dir.glob('*')):
    print(f'  {f.name}')

In [ ]:
# -- Read JSON schema example (Flavour B)
json_schema = read_json_schema(example_dir / 'olist_orders_schema_step2.json')

print(f'Table: {json_schema.table}')
print(f'Columns ({len(json_schema.columns)}):' )
for col in json_schema.columns:
    print(f'  [{col.position}] {col.column!r:40s} type={col.declared_type!r:10s} nullable={col.nullable}')

## 2. Name Normalizer

Normalisation, expansion d'abbréviations, préservation des termes métier.

In [ ]:
from intake.semantic_detection.step2.name_normalizer import NameNormalizer

norm = NameNormalizer()

test_columns = [
    'order_id',
    'ProductQty',
    'customer_zip_code_prefix',
    'review_comment_message',
    'geolocation_lat',
    'product_weight_g',
    'monthly_kpi',          # business term preserved
    'freight_value',
    'order_purchase_timestamp',
    'CustomerFirstName',
]

print(f'{"Column":<40} {"Normalized Key":<35} {"Tokens"}')
print('-' * 100)
for col in test_columns:
    result = norm.normalize(col)
    expanded = ', '.join(f'{k}→{v}' for k, v in result.expanded_abbreviations.items())
    print(f'{col:<40} {result.normalized_key:<35} {result.tokens}')
    if expanded:
        print(f'  ↳ Expansions: {expanded}')
    if result.preserved_business_terms:
        print(f'  ↳ Business terms: {result.preserved_business_terms}')

## 3. Rule-Based Classifier

Classification selon la taxonomie (5 tiers : alias → keyword → pattern → context → unknown).

In [ ]:
from intake.semantic_detection.step2.rule_classifier import RuleBasedClassifier
from intake.semantic_detection.step2.schema_reader import ColumnSchema, TableSchema

clf = RuleBasedClassifier()

# -- Classify a set of representative Olist columns
test_cols = [
    ('olist_orders',     'order_id'),
    ('olist_orders',     'customer_id'),
    ('olist_orders',     'order_status'),
    ('olist_orders',     'order_purchase_timestamp'),
    ('olist_order_items','price'),
    ('olist_order_items','freight_value'),
    ('olist_customers',  'customer_city'),
    ('olist_geolocation','geolocation_lat'),
    ('olist_geolocation','geolocation_zip_code_prefix'),
    ('olist_reviews',    'review_score'),
    ('olist_reviews',    'review_comment_message'),
    ('olist_products',   'product_weight_g'),
    ('any_table',        'totally_unknown_xz99'),   # Should be UNKNOWN
]

print(f'{"Table":<25} {"Column":<35} {"Group":<25} {"Type":<20} {"Conf":>5}  {"Tier"}')
print('-' * 130)
for table, col in test_cols:
    cs = ColumnSchema(table=table, column=col, position=0)
    r = clf.classify_column(cs)
    print(
        f'{r.table:<25} {r.column:<35} '
        f'{r.semantic_group:<25} {r.semantic_type:<20} '
        f'{r.confidence:>5.2f}  {r.match_tier}'
    )

In [ ]:
# -- Classify a full table with context
olist_geo = TableSchema(
    table='olist_geolocation',
    source_path='',
    source_format='csv',
    columns=[
        ColumnSchema(table='olist_geolocation', column='geolocation_zip_code_prefix', position=0),
        ColumnSchema(table='olist_geolocation', column='geolocation_lat',             position=1),
        ColumnSchema(table='olist_geolocation', column='geolocation_lng',             position=2),
        ColumnSchema(table='olist_geolocation', column='geolocation_city',            position=3),
        ColumnSchema(table='olist_geolocation', column='geolocation_state',           position=4),
    ]
)

results = clf.classify_table(olist_geo)
print('\nolist_geolocation table — contextual classification:')
for r in results:
    print(f'  {r.column:<35} → {r.semantic_type:<20} ({r.confidence:.2f}) [{r.match_tier}]')

## 4. Ambiguity Handler

Détection des colonnes inconnues, faible confiance, et résolution des conflits.

In [ ]:
from intake.semantic_detection.step2.ambiguity_handler import AmbiguityHandler, flag_unknown_columns, flag_low_confidence

handler = AmbiguityHandler()

# Classify all test columns and process for ambiguity
all_cols = [
    ColumnSchema(table=t, column=c, position=i)
    for i, (t, c) in enumerate(test_cols)
]
all_results = clf.classify_tables([olist_geo])

# Add the individually classified columns too
individual = []
for table, col in test_cols:
    cs = ColumnSchema(table=table, column=col, position=0)
    individual.append(clf.classify_column(cs))

resolved, report = handler.process(individual)

print('=== Ambiguity Report ===')
import json
print(json.dumps(report.summary(), indent=2))

if report.review_flags:
    print('\n=== Columns Requiring Review ===')
    for flag in report.review_flags:
        print(f'  [{flag.reason}] {flag.table}.{flag.column} (confidence={flag.confidence:.2f})')

## 5. Pipeline complet

Utilisation de `classify_directory()` pour traiter un répertoire complet de schemas CSV/JSON.

In [ ]:
from intake.semantic_detection.step2.pipeline import classify_directory

# Classify the examples directory (contains JSON schemas)
try:
    report, ambiguity = classify_directory(example_dir)
    print(f'ColumnTypesReport: model={report.model!r}, columns={len(report.columns)}')
    print(f'Ambiguity: {ambiguity.summary()}')
    print('\nClassified columns:')
    for col in report.columns:
        print(f'  {col.table}.{col.column:<35} → {col.semantic_type.value:<12} ({col.confidence:.2f}) pii={col.pii_candidate}')
except Exception as e:
    print(f'Note: {e}')
    print('(Expected if example dir only contains JSON files without CSV headers — works correctly with CSV dirs)')

## 6. Rapport d'évaluation — Olist Golden Set

Évaluation de la précision de l'Étape 2 sur les colonnes connues de l'Olist dataset.

In [ ]:
# Golden dataset — expected classifications for Olist
golden = {
    ('olist_orders', 'order_id'):              'ORDER_ID',
    ('olist_orders', 'customer_id'):           'PERSON_ID',
    ('olist_orders', 'order_status'):          'STATUS',
    ('olist_order_items', 'price'):            'PRICE',
    ('olist_order_items', 'freight_value'):    'FREIGHT_COST',
    ('olist_customers', 'customer_city'):      'CITY',
    ('olist_customers', 'customer_state'):     'STATE',
    ('olist_geolocation', 'geolocation_lat'):  'LATITUDE',
    ('olist_geolocation', 'geolocation_lng'):  'LONGITUDE',
    ('olist_reviews', 'review_score'):         'RATING',
    ('olist_reviews', 'review_comment_message'): 'COMMENT',
    ('olist_products', 'product_weight_g'):    'WEIGHT',
}

correct = 0
total = len(golden)
print(f'{"Table.Column":<45} {"Expected":<20} {"Got":<20} {"Match"}')
print('-' * 100)
for (table, col), expected_type in golden.items():
    cs = ColumnSchema(table=table, column=col, position=0)
    result = clf.classify_column(cs)
    match = result.semantic_type == expected_type
    if match:
        correct += 1
    status = '✅' if match else '❌'
    print(f'{table+"."+col:<45} {expected_type:<20} {result.semantic_type:<20} {status}')

accuracy = correct / total * 100
print(f'\n=== Accuracy: {correct}/{total} = {accuracy:.1f}% ===')

---
## Résumé

| Composant | Statut | Description |
|-----------|--------|-------------|
| Schema Reader | ✅ | CSV + JSON (3 flavours) |
| Name Normalizer | ✅ | 70+ abbréviations, camelCase, business terms |
| Rule Classifier | ✅ | 5 tiers, 50+ règles YAML |
| Ambiguity Handler | ✅ | Unknown, low-conf, conflict resolution |
| Pipeline | ✅ | DAG entrypoint + standalone helper |
| Tests | ✅ | 117 tests across 4 files |